In [0]:
import os
import base64
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ObjectType, ExportFormat, ImportFormat
w = WorkspaceClient()
catalog = dbutils.widgets.get("catalog")
schemas = dbutils.widgets.get("schemas")

# Replace placeholders in generated notebooks
for schema in schemas:
    output_folder = f"/Workspace/Users/depoplimontj@delawareconsulting.com/delaware_lakebridge/switch_runs/{schema}" 
    for obj in sorted(w.workspace.list(output_folder), key=lambda o: o.path):
        if obj.object_type == ObjectType.NOTEBOOK:
            export_resp = w.workspace.export(obj.path, format=ExportFormat.SOURCE)
            content = base64.b64decode(export_resp.content).decode("utf-8")
            content = content.replace("{catalog}", catalog).replace("{schema}", schema)
            w.workspace.import_(
                obj.path,
                content=base64.b64encode(content.encode("utf-8")).decode("utf-8"),
                format=ImportFormat.SOURCE,
                language=obj.language,
                overwrite=True,
            )

    # Run generated notebooks
    for obj in sorted(w.workspace.list(output_folder), key=lambda o: o.path):
        if obj.object_type == ObjectType.NOTEBOOK:
            dbutils.notebook.run(
                obj.path,
                timeout_seconds=100000,
                arguments={"catalog": catalog, "schema": schema},
            )